# Manual Labeling Helper
This notebook applies conservative rule-based auto-labeling to CSV files located in `../output` (relative to this notebook).
It replaces `Label` values equal to `NeedManualLabel` with `Benign` or `Attack` based on simple heuristics, and writes a backup before overwriting.

In [1]:
# Rule-based auto-labeling helper
import pandas as pd
from pathlib import Path
import shutil, time, os
import warnings
warnings.simplefilter('ignore')

# Helper to find the best matching column name from a list of candidates
def get_col(df, candidates):
    cn = [c for c in candidates if c in df.columns]
    return cn[0] if cn else None

# Conservative inference rules: return (label, reason)
def infer_label_from_row(row):
    # Accept many possible column names for robustness
    proto = None
    for k in ['Protocol','protocol','Proto']:
        if k in row.index:
            proto = str(row[k]).lower() if pd.notna(row[k]) else None
            break

    dst_port = None
    for k in ['Dst Port','DstPort','dst_port','dport','destination_port','Destination Port','DstPort']:
        if k in row.index:
            try:
                dst_port = int(float(row[k]))
            except Exception:
                dst_port = None
            break

    flow_dur = None
    for k in ['Flow Duration','flow_duration','flowDuration']:
        if k in row.index:
            try:
                flow_dur = float(row[k])
            except Exception:
                flow_dur = None
            break

    tot_pkts = None
    for k in ['Tot Fwd Pkts','Total Fwd Packets','total_fwd_packets','tot_fwd_pkts','Fwd Packets','Total Packets']:
        if k in row.index:
            try:
                tot_pkts = int(float(row[k]))
            except Exception:
                tot_pkts = None
            break

    tot_bytes = None
    for k in ['Flow Bytes','Total Length','TotLen Fwd Pkts','Total Fwd Bytes','flow_bytes','tot_bytes']:
        if k in row.index:
            try:
                tot_bytes = float(row[k])
            except Exception:
                tot_bytes = None
            break

    # Derived features
    avg_pkt_len = None
    if tot_bytes is not None and tot_pkts not in (None, 0):
        try:
            avg_pkt_len = tot_bytes / max(1, tot_pkts)
        except Exception:
            avg_pkt_len = None

    # Conservative rule set (aim to be helpful but avoid false positives)
    # Rule 1: ICMP floods or many small packets -> attack
    if proto and 'icmp' in proto and (tot_pkts is not None and tot_pkts > 100):
        return 'Attack', 'ICMP large packet count'

    # Rule 2: Very high packet count in short duration (DDoS-like)
    if tot_pkts is not None and flow_dur not in (None, 0) and tot_pkts > 1000:
        # flow_dur in microseconds or ms depending on dataset; choose conservative check
        if flow_dur < 1000000:  # less than ~1000s (very permissive)
            return 'Attack', 'High packets in short flow duration'

    # Rule 3: Known amplification ports with moderate packet counts -> attack
    amp_ports = {53,123,1900,161,137,138}
    if dst_port in amp_ports and (tot_pkts is not None and tot_pkts > 50):
        return 'Attack', f'Amplification-like dst port {dst_port}'

    # Rule 4: Extremely large average packet size -> suspicious (possible data exfil or unusual traffic)
    if avg_pkt_len is not None and avg_pkt_len > 10000:
        return 'Attack', 'Very large average packet length'

    # Default: mark benign
    return 'Benign', 'Default conservative rule'

def auto_label_df(df, verbose=False):
    # Find label column
    label_col = get_col(df, ['Label','label','PredictedLabel','predicted_label'])
    if label_col is None:
        raise RuntimeError('No label column found in dataframe')

    # Prepare reason column
    reason_col = 'AutoLabelReason'
    if reason_col not in df.columns:
        df[reason_col] = ''

    mask = df[label_col].astype(str) == 'NeedManualLabel'
    if verbose:
        print(f'Total rows: {len(df)}, candidates to auto-label: {mask.sum()}')

    changed = 0
    for idx in df[mask].index:
        try:
            label, reason = infer_label_from_row(df.loc[idx])
            df.at[idx, label_col] = label
            df.at[idx, reason_col] = reason
            changed += 1
        except Exception as e:
            if verbose:
                print('Error inferring row', idx, e)
            continue

    return df, changed

def find_output_dir():
    # This notebook lives in backend/playground; output is typically backend/output
    cwd = Path.cwd()
    candidates = [cwd / 'backend' / 'output', cwd / 'output', cwd.parent / 'output']
    for c in candidates:
        if c.exists() and c.is_dir():
            return c
    raise FileNotFoundError('Could not find output directory; tried: ' + ','.join(str(x) for x in candidates))

In [2]:
# Run auto-labeling across CSVs in output/ (creates backups)
out_dir = find_output_dir()
print('Using output directory:', out_dir)
csvs = sorted([p for p in out_dir.iterdir() if p.suffix.lower() == '.csv'])
if not csvs:
    print('No CSVs found in', out_dir)

summary = {}
for f in csvs:
    try:
        print('Processing', f.name)
        # Make a backup copy first
        ts = int(time.time())
        backup = f.with_suffix(f.suffix + f'.backup_{ts}')
        if not backup.exists():
            shutil.copy2(f, backup)
        # Read CSV (pandas will raise for LFS pointer files if not resolved)
        df = pd.read_csv(f)
        before_need = df[get_col(df, ['Label','label','PredictedLabel','predicted_label'])].astype(str).eq('NeedManualLabel').sum()
        df2, changed = auto_label_df(df, verbose=False)
        df2.to_csv(f, index=False)
        summary[f.name] = {'candidates': int(before_need), 'changed': int(changed), 'backup': backup.name}
        print(f"  candidates={before_need}, changed={changed}, backup={backup.name}")
    except pd.errors.EmptyDataError:
        print('  empty or unreadable CSV (maybe a git-lfs pointer):', f.name)
        summary[f.name] = {'error':'unreadable_or_empty'}
    except Exception as e:
        print('  error processing', f.name, e)
        summary[f.name] = {'error': str(e)}

print('\nSummary:')
for k,v in summary.items():
    print(' -', k, v)

Using output directory: /home/fyp2025/fyp/backend/output
Processing mirror.pcap_Flow_predicted.csv
  error processing mirror.pcap_Flow_predicted.csv None
Processing mirror10min.pcap_Flow_predicted.csv
  error processing mirror10min.pcap_Flow_predicted.csv None
Processing mirror_predicted.csv
  error processing mirror_predicted.csv None
Processing nightmirror_predicted.csv
  candidates=534024, changed=534024, backup=nightmirror_predicted.csv.backup_1764579692

Summary:
 - mirror.pcap_Flow_predicted.csv {'error': 'None'}
 - mirror10min.pcap_Flow_predicted.csv {'error': 'None'}
 - mirror_predicted.csv {'error': 'None'}
 - nightmirror_predicted.csv {'candidates': 534024, 'changed': 534024, 'backup': 'nightmirror_predicted.csv.backup_1764579692'}
  candidates=534024, changed=534024, backup=nightmirror_predicted.csv.backup_1764579692

Summary:
 - mirror.pcap_Flow_predicted.csv {'error': 'None'}
 - mirror10min.pcap_Flow_predicted.csv {'error': 'None'}
 - mirror_predicted.csv {'error': 'None'}

In [10]:
import torch

print(torch.cuda.get_device_name())

NVIDIA GeForce RTX 3080 Ti
